# Exemple d'utilisation des scripts ASN queries

Ce notebook montre comment utiliser les modules Python du dossier `scripts/`.

## 1. Configuration des chemins et imports

In [1]:
import sys
from pathlib import Path

# Le notebook est dans asn_queries/, donc scripts/ est directement accessible
# Utiliser le répertoire courant comme base
asn_queries_dir = Path.cwd()
scripts_dir = asn_queries_dir / "scripts"

# Ajouter le dossier parent (asn_queries/) au PYTHONPATH
# Cela permet d'importer scripts comme un package
sys.path.insert(0, str(asn_queries_dir))

# Ajouter aussi scripts/ au PYTHONPATH pour les imports absolus comme stats, references
sys.path.insert(0, str(scripts_dir))

print(f"✅ Dossier asn_queries : {asn_queries_dir}")
print(f"✅ Dossier scripts : {scripts_dir}")

✅ Dossier asn_queries : /home/jovyan/work/benchmark/asn_queries
✅ Dossier scripts : /home/jovyan/work/benchmark/asn_queries/scripts


## 2. Import des modules

In [2]:
# Importer les modules du package scripts
from scripts import io, asn, paths
from scripts import stats, references

# Importer RMatrix
from matrix_bgpsim import RMatrix

print("✅ Modules importés avec succès")

✅ Modules importés avec succès


## 3. Chargement de la matrice RMatrix

In [3]:
# Initialiser le dossier queries
io.ensure_query_dir()
io.move_legacy_files()

# Charger le fichier .lz4 depuis scripts/
rmatrix_file = scripts_dir / "rmatrix-cupy-core-20000.20250101.as-rel2.lz4"
if not rmatrix_file.exists():
    # Essayer depuis le répertoire parent (benchmark/)
    rmatrix_file = asn_queries_dir.parent / "rmatrix-cupy-core-20000.20250101.as-rel2.lz4"

print(f"📂 Chargement de : {rmatrix_file}")
if not rmatrix_file.exists():
    print(f"❌ Fichier introuvable : {rmatrix_file}")
else:
    rmatrix = RMatrix.load(rmatrix_file)
    print("✅ Matrice chargée avec succès")

📂 Chargement de : /home/jovyan/work/benchmark/asn_queries/scripts/rmatrix-cupy-core-20000.20250101.as-rel2.lz4
✅ Matrice chargée avec succès


## 4. Construction/chargement des listes d'ASN

In [4]:
# Construire ou charger les listes d'ASN (présents, core, branch)
asn_present, asn_core, asn_branch = asn.build_or_load_asn(rmatrix)

print(f"\n📊 Résumé :")
print(f"   - ASN présents : {len(asn_present)}")
print(f"   - ASN core : {len(asn_core)}")
print(f"   - ASN branch : {len(asn_branch)}")

✅ Nombre d'ASN présents : 7206
♥️  Nombre total de Core AS : 7206
🌿 Nombre total de Branch AS : 0

📊 Résumé :
   - ASN présents : 7206
   - ASN core : 7206
   - ASN branch : 0


## 5. Exemple : Top 10 AS les plus connectés

In [5]:
# Obtenir les 10 AS les plus connectés
top_connected = stats.get_top_connected_as(rmatrix, top_n=10)

print("🔗 Top 10 AS les plus connectés :\n")
for rank, (asn_val, connections) in enumerate(top_connected, 1):
    print(f"  {rank:2d}. AS{asn_val:>8s} : {connections:4d} relations")

🔗 Top 10 AS les plus connectés :

   1. AS     174 : 6738 relations
   2. AS    6939 : 5978 relations
   3. AS    3356 : 5851 relations
   4. AS   24482 : 5089 relations
   5. AS    1239 : 4578 relations
   6. AS   39120 : 4131 relations
   7. AS  199524 : 4066 relations
   8. AS   49544 : 4008 relations
   9. AS   35280 : 3819 relations
  10. AS  271253 : 3442 relations


## 6. Exemple : Chemin le plus long

In [6]:
# Charger ou calculer le chemin le plus long
src, dst, path, length = paths.load_or_compute_longest_path(
    rmatrix, list(asn_core), list(asn_present)
)

if path:
    print(f"🗺️  Chemin le plus long :")
    print(f"   Source : AS{src}")
    print(f"   Destination : AS{dst}")
    print(f"   Longueur : {length} AS intermédiaires")
    print(f"   Chemin : AS{src} -> {' -> '.join(path)} -> AS{dst}")
else:
    print("❌ Aucun chemin trouvé")

🗺️  Chemin le plus long :
   Source : AS15808
   Destination : AS266022
   Longueur : 15 AS intermédiaires
   Chemin : AS15808 -> 327727 -> 327814 -> 37613 -> 37645 -> 13335 -> 20473 -> 52025 -> 199524 -> 52925 -> 272713 -> 28189 -> 267521 -> 53043 -> 263433 -> 266022 -> AS266022


## 7. Exemple : Statistiques complètes de la topologie

In [7]:
# Afficher toutes les statistiques de la topologie
stats.compute_topology_stats(rmatrix, list(asn_core), list(asn_present))

📊 Statistiques de la topologie

🔗 Top 10 AS les plus connectés :

   1. AS     174 : 6738 relations totales (P2C: 6640, P2P:  98, C2P:   0)
   2. AS    6939 : 5978 relations totales (P2C: 1540, P2P: 4435, C2P:   3)
   3. AS    3356 : 5851 relations totales (P2C: 5785, P2P:  66, C2P:   0)
   4. AS   24482 : 5089 relations totales (P2C:  35, P2P: 5049, C2P:   5)
   5. AS    1239 : 4578 relations totales (P2C: 2942, P2P: 1636, C2P:   0)
   6. AS   39120 : 4131 relations totales (P2C:   2, P2P: 4125, C2P:   4)
   7. AS  199524 : 4066 relations totales (P2C:  14, P2P: 4007, C2P:  45)
   8. AS   49544 : 4008 relations totales (P2C:   7, P2P: 3994, C2P:   7)
   9. AS   35280 : 3819 relations totales (P2C:  31, P2P: 3782, C2P:   6)
  10. AS  271253 : 3442 relations totales (P2C:  40, P2P: 3396, C2P:   6)

🔀 AS à l'intersection de plusieurs chemins :

  AS265269 apparaît dans 5 chemins différents

  Chemins contenant AS265269 :

  Chemin 1 (AS266616 -> AS205147) :
266616___265269___174___205147

## 8. Exemple : Mise à jour des références

In [8]:
# Mettre à jour le fichier references.txt
references.update_references()

✅ Fichier /home/jovyan/work/benchmark/asn_queries/scripts/references.txt mis à jour avec 13 types de références trouvées.
